In [1]:
%load_ext autoreload
%autoreload 2
 
import itertools
 
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from tqdm.auto import tqdm
 
from assignment_1.modules.models import BPRModel, PairFeatures, binary_cosine, jaccard
from assignment_1.modules.tasks.task4 import TASK4_BPR_PARAMS
from assignment_1.modules.util import calc_pair_accuracy, load_csv, make_pair_validation

In [2]:
DATA_DIR = "../data"  # notebook runs from notebooks/
 
train = load_csv("tasks_3_4_train.csv", DATA_DIR)
test = load_csv("task_4_test.csv", DATA_DIR)

In [3]:
first_seen = train.groupby("item")["timestamp"].min()
early1 = test["item1"].map(first_seen) > test["timestamp"]
early2 = test["item2"].map(first_seen) > test["timestamp"]
print(f"test: item1 not-yet-seen {early1.mean():.2%}, item2 {early2.mean():.2%}, "
      f"exactly one {(early1 ^ early2).mean():.2%}")

test: item1 not-yet-seen 12.97%, item2 13.56%, exactly one 25.47%


In [4]:
tr, pairs = make_pair_validation(train, include_timestamp=True, seed=0)
y = (pairs["which"] == pairs["item1"]).astype(int).to_numpy()
which = pairs["which"].to_numpy()
half = np.random.default_rng(0).random(len(pairs)) < 0.5
 
fs_tr = tr.groupby("item")["timestamp"].min()
e1 = pairs["item1"].map(fs_tr) > pairs["timestamp"]
e2 = pairs["item2"].map(fs_tr) > pairs["timestamp"]
one = (e1 ^ e2).to_numpy()
seen_item = np.where(e1, pairs["item2"], pairs["item1"])
print(f"{len(pairs):,} pairs | exactly one not-yet-seen: {one.mean():.2%} | "
      f"rule accuracy there: {(seen_item[one] == which[one]).mean():.3f}")

142,924 pairs | exactly one not-yet-seen: 26.59% | rule accuracy there: 0.970


In [5]:
bpr = BPRModel(**TASK4_BPR_PARAMS).fit(tr)
users = pairs["user"].to_numpy()
s1, s2 = bpr(users, pairs["item1"].to_numpy()), bpr(users, pairs["item2"].to_numpy())
rule_pred = np.where(one, seen_item, np.where(s1 >= s2, pairs["item1"], pairs["item2"]))
print(f"BPR + rule, held-out half: {(rule_pred[~half] == which[~half]).mean():.4f}  (expect 0.8803)")

BPR + rule, held-out half: 0.8803  (expect 0.8803)


In [6]:
def bpr_grid(grid: dict) -> pd.DataFrame:
    rows = []
    for values in tqdm(list(itertools.product(*grid.values()))):
        params = dict(zip(grid, values))
        b = BPRModel(**params).fit(tr)
        p = np.where(one, seen_item, np.where(b(users, pairs["item1"].to_numpy())
                                              >= b(users, pairs["item2"].to_numpy()),
                                              pairs["item1"], pairs["item2"]))
        rows.append({**params, "bpr_acc": calc_pair_accuracy(pairs, b), "combined_acc": (p == which).mean()})
    return pd.DataFrame(rows).sort_values("combined_acc", ascending=False).reset_index(drop=True)

In [7]:
# bpr_grid({"factors": [50, 100, 200], "regularization": [0.003, 0.01, 0.03],
#           "learning_rate": [0.01], "iterations": [100]})

In [8]:
def evaluate(X: np.ndarray) -> dict:
    out = {}
    for name, m in [("lr", LogisticRegression(max_iter=1000)),
                    ("gb", HistGradientBoostingClassifier(random_state=0))]:
        m.fit(X[half], y[half])
        out[name] = round((m.predict(X[~half]) == y[~half]).mean(), 4)
    return out

In [9]:
X = PairFeatures(TASK4_BPR_PARAMS).fit(tr).transform(pairs)
evaluate(X)  

{'lr': np.float64(0.9018), 'gb': np.float64(0.9041)}

In [10]:
variants = {
    "shipped (no similarity)": None,
    "cosine": (binary_cosine, "cos"),
    "jaccard": (jaccard, "jac"),
}
for name, spec in variants.items():
    pf = (PairFeatures(TASK4_BPR_PARAMS) if spec is None
          else PairFeatures(TASK4_BPR_PARAMS, similarity=spec[0], sim_prefix=spec[1]))
    print(f"{name:<25}", evaluate(pf.fit(tr).transform(pairs)))

shipped (no similarity)   {'lr': np.float64(0.9018), 'gb': np.float64(0.9041)}
cosine                    {'lr': np.float64(0.9136), 'gb': np.float64(0.9173)}
jaccard                   {'lr': np.float64(0.9127), 'gb': np.float64(0.9158)}
